In [ ]:
# Block 1: notebook description and analysis objective

# This notebook isolates single-asset factor attribution and idiosyncratic risk.
# Extracted from Momentum & Efficiency.ipynb.


In [ ]:
import logging
import warnings
from pathlib import Path
import runpy

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
import yfinance as yf
from plotly.subplots import make_subplots

from Quantapp.visualization import Plotter
from Quantapp.analytics import Models

warnings.filterwarnings("ignore")
logger = logging.getLogger("yfinance")

In [ ]:
logger.disabled = True
logger.propagate = False

qp = Plotter()
model = Models()

PARAMS_FILE_CANDIDATES = []
for base_path in (Path.cwd(), *Path.cwd().parents):
    PARAMS_FILE_CANDIDATES.extend([
        base_path / "params.py",
        base_path / "Notebooks" / "Single Asset Profile" / "params.py",
    ])

seen_params_paths = set()
for params_file in PARAMS_FILE_CANDIDATES:
    params_file = params_file.resolve()
    if params_file in seen_params_paths:
        continue
    seen_params_paths.add(params_file)
    if params_file.exists():
        break
else:
    raise FileNotFoundError("Could not locate params.py in the Single Asset Profile workspace.")

notebook_params = runpy.run_path(str(params_file))
get_factor_attribution_params = notebook_params["get_factor_attribution_params"]


PLOTLY_NOTEBOOK_CONFIG = {"responsive": True, "scrollZoom": True}
for renderer_name in ("plotly_mimetype", "notebook", "notebook_connected", "jupyterlab"):
    try:
        pio.renderers[renderer_name].config = PLOTLY_NOTEBOOK_CONFIG.copy()
    except Exception:
        pass

def show_plotly_figure(fig, *, config=None, **layout_kwargs):
    merged_config = PLOTLY_NOTEBOOK_CONFIG.copy()
    if config:
        merged_config.update(config)
    fig.update_layout(autosize=True, **layout_kwargs)
    fig.show(config=merged_config)

In [ ]:
# Block 3: load shared factor attribution parameters
# Edit Notebooks/Single Asset Profile/params.py to update these defaults for every notebook.
factor_params = get_factor_attribution_params()

ticker_str = factor_params["ticker_str"]
interval = factor_params["interval"]
analysis_period = factor_params["analysis_period"]
rolling_window = factor_params["rolling_window"]
ff_verbose = factor_params["ff_verbose"]

factor_params

In [ ]:
# Block 4: run factor attribution and idiosyncratic risk analysis
ff_proxy = model.run_ff5_proxy_analysis(
    asset_ticker=ticker_str,
    period=analysis_period,
    interval=interval,
    window=rolling_window,
    auto_window=True,
    verbose=ff_verbose,
)

prices_df = ff_proxy["proxy_prices"]
returns = ff_proxy["proxy_returns"]
factor_returns = ff_proxy["factor_returns_all"]
factor_returns_capm = ff_proxy["factor_returns_capm"]
factor_returns_ff3 = ff_proxy["factor_returns_ff3"]
factor_returns_ff5 = ff_proxy["factor_returns_ff5"]
factor_returns_ff5_custom = factor_returns_ff5.copy()
stock_returns = ff_proxy["stock_returns"]
rolling_results_ff5_custom = ff_proxy["rolling_results"]

# Plot the Fama-French factor analysis results
ff_figs = qp.plot_rolling_regression(
    rolling_results_ff5_custom,
    ticker_str,
    factor_returns_ff5_custom,
    show=False,
)
show_plotly_figure(ff_figs["alpha"])
show_plotly_figure(ff_figs["betas"])
show_plotly_figure(ff_figs["r_squared"])

idio_fig = qp.plot_idiosyncratic_risk(
    rolling_results_ff5_custom,
    ticker_str,
    template="plotly_dark",
    show=False,
)
show_plotly_figure(idio_fig)
